# 05 · Local LLMs: Ollama, memory sizing, and running a model yourself

After this notebook you can install and drive Ollama on Windows, call it through its own REST API *and* through the unchanged OpenAI SDK, estimate how much RAM a model needs before you download it, pick a quantisation level, and run a small chat model locally with Hugging Face `transformers`, with its speed measured.

**Time:** ~30 min · **Runs:** offline on CPU in about a minute (Ollama cells skip cleanly if Ollama isn't running) · **Needs:** [01 · OpenAI API basics](01_openai_api_basics.ipynb). Background: [quantization notebook](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)

## Why this matters in interviews

- "Would you self-host or use an API?" comes up in almost every system-design round. They want the real reasons (data residency, volume, latency control) and a cost crossover, not a preference.
- "How much memory does this model need?" is a whiteboard calculation: parameters × bytes per parameter + KV cache + overhead. You'll do it here in code and check it against real download sizes.
- Knowing that Ollama speaks the OpenAI API, and where the compatibility breaks, is exactly the kind of practical detail that separates people who've shipped from people who've read about it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl11` | What does Ollama do, and would you use it in production? |
| `tl12` | When does self-hosting a model actually beat a hosted API? |
| `in19` | When would you self-host a model instead of using an API? |
| `tl19` | Design the tooling for a regulated enterprise that cannot send data to a third party. |
| `in13` | Which inference server would you choose and why? |
| `in07` | What is quantisation and which method would you pick? |
| `tr29` | How do you compute the memory a model needs to serve? |
| `tn24` | What is a chat template and why does it matter when self-hosting? |
| `in37` | What is tokens-per-second and how do you interpret it? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import json, os, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from llm_setup import ollama_running, OLLAMA_URL, OLLAMA_DEFAULT_MODEL

OLLAMA_UP = ollama_running()
print(f"Ollama at {OLLAMA_URL}: {'running' if OLLAMA_UP else 'not running (Ollama cells will skip)'}")
print("notebook provider:", PROVIDER, "| model:", MODEL)

## 1. Why run a model locally?

| Reason | What it buys you | The catch |
|---|---|---|
| **Privacy / data residency** | prompts never leave the machine or the VPC. For regulated data this decides the question outright | you now own security patching and access control |
| **Cost** | no per-token bill: free for development, cheap at *steady high* volume | GPUs cost money when idle, plus engineering and on-call time. Bursty traffic is the worst fit |
| **Offline / air-gapped** | works on a plane, in a factory, in a secure enclave | model updates are manual |
| **Latency control** | no network hop, no shared-tenant queue | small local hardware is often *slower* per token than a big hosted GPU |
| **Customisation** | your own fine-tunes, any open model, full control of sampling | quality gap vs frontier models on hard reasoning |

The honest summary: local models are superb for development, privacy-bound workloads, and narrow tasks at high volume. For low-volume, high-difficulty work a hosted API is usually cheaper and better once you count engineering time.

## 2. Installing and driving Ollama (Windows)

**In plain English:** Ollama is a small program that downloads open models in a compressed format (GGUF) and serves them on `http://localhost:11434`, with its own API *and* an OpenAI-compatible one. Under the hood it runs llama.cpp.

1. Install: download `OllamaSetup.exe` from [ollama.com/download](https://ollama.com/download), or run `winget install --id Ollama.Ollama`. It starts a background server and a tray app. Models are stored in `%USERPROFILE%\.ollama` (move them with the `OLLAMA_MODELS` environment variable).
2. Pull the models these notebooks default to: `ollama pull qwen2.5:3b` (~1.9 GB) and `ollama pull nomic-embed-text` (~0.3 GB).
3. Then `llm_setup.py` finds it automatically. There's nothing to configure.

| Command | What it does |
|---|---|
| `ollama pull qwen2.5:3b` | download a model (name:tag, the tag picks size and quantisation) |
| `ollama run qwen2.5:3b` | interactive chat in the terminal (`/bye` to quit) |
| `ollama run qwen2.5:3b "Say hi"` | one-shot prompt |
| `ollama list` | models on disk, with sizes |
| `ollama ps` | models loaded in memory right now: size, CPU/GPU split, context length |
| `ollama show qwen2.5:3b` | architecture, parameters, context length, quantisation, chat template |
| `ollama rm qwen2.5:3b` | delete a model |
| `ollama serve` | start the server by hand (e.g. with `OLLAMA_HOST=127.0.0.1:11500` for another port) |

## 3. Ollama's own REST API

Three endpoints cover most use: `GET /api/tags` (installed models), `POST /api/chat` (messages in, message out), `POST /api/generate` (raw prompt in, text out). These cells only run if Ollama is up. If it isn't, they print how to start it, which is the one legitimate "if the service is there" branch in these notebooks.

In [ ]:
def ollama_get(path):
    with urllib.request.urlopen(OLLAMA_URL.rstrip("/") + path, timeout=10) as r:
        return json.loads(r.read())

def ollama_post(path, body, timeout=300):
    req = urllib.request.Request(OLLAMA_URL.rstrip("/") + path, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

LOCAL_MODEL = None
if OLLAMA_UP:
    installed = {m["name"]: m for m in ollama_get("/api/tags")["models"]}
    for name, m in installed.items():
        d = m.get("details", {})
        print(f"{name:<24} {m['size'] / 1e9:5.2f} GB  params={d.get('parameter_size')}  quant={d.get('quantization_level')}")
    wanted = os.getenv("OLLAMA_MODEL", OLLAMA_DEFAULT_MODEL)
    LOCAL_MODEL = wanted if wanted in installed else next(iter(installed), None)
    if LOCAL_MODEL != wanted:
        print(f"\n{wanted} is not pulled (run: ollama pull {wanted}); using {LOCAL_MODEL} for the demos")
else:
    print("Ollama is not running. Install it (section 2), run `ollama pull qwen2.5:3b`, then re-run this notebook.")

In [ ]:
if OLLAMA_UP and LOCAL_MODEL:
    r = ollama_post("/api/chat", {
        "model": LOCAL_MODEL, "stream": False,
        "messages": [{"role": "user", "content": "What is the capital of France? Answer in one word."}],
        "options": {"temperature": 0, "num_predict": 20},          # num_predict = max new tokens
    })
    print("reply        :", r["message"]["content"].strip())
    print("load time    :", round(r["load_duration"] / 1e9, 2), "s  (a cold start: first call loads the model into RAM)")
    print("prompt       :", r["prompt_eval_count"], "tokens at",
          round(r["prompt_eval_count"] / max(r["prompt_eval_duration"], 1) * 1e9), "tok/s (prefill)")
    print("generation   :", r["eval_count"], "tokens at",
          round(r["eval_count"] / max(r["eval_duration"], 1) * 1e9), "tok/s (decode)")
    loaded = ollama_get("/api/ps")["models"]
    print("loaded now   :", [(m["name"], f"{m['size'] / 1e9:.2f} GB", f"ctx={m.get('context_length')}") for m in loaded])
else:
    print("skipped: Ollama not running")

What the `/api/chat` response tells you, for free, on every call: `load_duration` (cold start), `prompt_eval_count/duration` (prefill speed) and `eval_count/duration` (decode speed), all in nanoseconds. Prefill is much faster per token than decode, because the prompt is processed in parallel while output is produced one token at a time. `/api/generate` is the same with `"prompt": "..."` instead of messages. Pass `"stream": true` (the default) to get one JSON object per line as tokens arrive.

## 4. The same OpenAI SDK code, pointed at Ollama

Ollama also serves `/v1/chat/completions`, `/v1/responses`, `/v1/embeddings` and `/v1/models`. **Only `base_url` changes**, and the key can be any non-empty string:

In [ ]:
from openai import OpenAI

if OLLAMA_UP and LOCAL_MODEL:
    local = OpenAI(base_url=OLLAMA_URL.rstrip("/") + "/v1", api_key="ollama")
    r = local.chat.completions.create(model=LOCAL_MODEL, temperature=0, max_tokens=40, messages=[
        {"role": "system", "content": "Answer in one short sentence."},
        {"role": "user", "content": "Why is the sky blue?"}])
    print(r.choices[0].message.content)
    print("usage:", r.usage.prompt_tokens, "in /", r.usage.completion_tokens, "out | finish:", r.choices[0].finish_reason)
else:
    print("skipped: Ollama not running")

That's why every other notebook here works with Ollama unchanged. `llm_setup.py` decides the provider like this:

```python
def _pick_provider():
    wanted = os.getenv("LLM_PROVIDER", "")          # 1. explicit choice wins: openai | ollama | offline
    ...
    if os.getenv("OPENAI_API_KEY"): return "openai"  # 2. a key means OpenAI
    if ollama_running():            return "ollama"  # 3. GET {OLLAMA_URL}/api/tags answers within 0.5 s
    return "offline"                                 # 4. the deterministic stand-in
```

For Ollama it builds `OpenAI(base_url=OLLAMA_URL + "/v1", api_key="ollama")` with model `qwen2.5:3b` (override with `OLLAMA_MODEL`). To force Ollama while you also have a key, set `LLM_PROVIDER=ollama` in `.env`.

**Where "OpenAI-compatible" stops.** These were checked against Ollama 0.34 and may change in later versions:

| Feature | OpenAI | Ollama `/v1` |
|---|---|---|
| `max_tokens` | legacy, still accepted by non-reasoning models | **the only cap it honours** |
| `max_completion_tokens` | current name | **silently ignored**: a "5-token" test generated 1,000+ tokens |
| `n` (several samples) | yes | ignored: you always get 1 choice |
| `tool_choice` | yes | not supported |
| tools | yes | only for models whose template supports them (`qwen2.5` yes, `gemma3:270m` returns a 400) |
| structured outputs (`.parse`) | yes | yes |
| `logprobs` | yes | listed as unsupported in Ollama's docs |
| unknown model | 404 | 404 `model 'x' not found`: you forgot `ollama pull` |
| default context | model maximum | **4k tokens** on machines with < 24 GB VRAM. Longer prompts are truncated, so raise it with `OLLAMA_CONTEXT_LENGTH` |

## 5. How much memory does a model need?

**In plain English:** memory = the weights (parameters × bytes per parameter) + the KV cache (a per-token scratchpad that grows with context length) + a bit of runtime overhead.

- **Weights:** fp16 = 2 bytes/param, Q8_0 ≈ 1.06, Q5_K_M ≈ 0.7, **Q4_K_M ≈ 0.6** (4-bit blocks plus their scales, and some tensors kept at higher precision).
- **KV cache:** `2 (K and V) × layers × kv_heads × head_dim × context × bytes` (fp16 = 2 bytes). Grouped-query attention (few KV heads) is why modern models keep this small.

The architecture numbers below follow the Qwen2.5 family's published configs.

In [ ]:
MODELS = pd.DataFrame([
    # name, params (billions), layers, kv_heads, head_dim, Ollama q4_K_M download (GB, ollama.com/library)
    ("0.5B", 0.49, 24, 2, 64, 0.398),
    ("1.5B", 1.54, 28, 2, 128, 0.986),
    ("3B",   3.09, 36, 2, 128, 1.9),
    ("7B",   7.62, 28, 4, 128, 4.7),
    ("14B",  14.7, 48, 8, 128, 9.0),
], columns=["model", "params_b", "layers", "kv_heads", "head_dim", "ollama_q4_gb"]).set_index("model")

BYTES_PER_PARAM = {"fp16": 2.0, "Q8_0": 1.06, "Q5_K_M": 0.71, "Q4_K_M": 0.60}

def kv_cache_gb(layers, kv_heads, head_dim, context, kv_bytes=2):
    return 2 * layers * kv_heads * head_dim * context * kv_bytes / 1e9

def ram_gb(row, quant="Q4_K_M", context=4096, overhead_gb=0.5):
    weights = row.params_b * BYTES_PER_PARAM[quant]
    kv = kv_cache_gb(row.layers, row.kv_heads, row.head_dim, context)
    return weights, kv, overhead_gb

est = pd.DataFrame([ram_gb(row) for row in MODELS.itertuples()], index=MODELS.index,
                   columns=["weights_gb", "kv_4k_gb", "overhead_gb"])
est["total_gb"] = est.sum(axis=1)
est["ollama_download_gb"] = MODELS["ollama_q4_gb"]
est["weights_vs_download"] = est["weights_gb"] / est["ollama_download_gb"] - 1
print(est.round(2).to_string())

big = est.loc[["1.5B", "3B", "7B", "14B"], "weights_vs_download"].abs()
assert (big < 0.10).all(), "the 0.6 bytes/param rule should be within 10% of real Q4_K_M files"

The rule of thumb is within a few percent of the real Q4_K_M files for 1.5B and up. For the 0.5B model it undershoots, because its embedding table (a 150k-token vocabulary) is a big share of such a small model and is stored at higher precision. So: **a 7B model at Q4 needs ~5.5 GB of RAM at a 4k context**, which fits a 16 GB laptop comfortably, and a 14B model needs ~10.5 GB.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
left = np.zeros(len(est))
for col, color, label in [("weights_gb", "#1c7ed6", "weights (Q4_K_M)"), ("kv_4k_gb", "#e8590c", "KV cache @ 4k context"),
                          ("overhead_gb", "#adb5bd", "runtime overhead")]:
    ax.barh(est.index, est[col], left=left, color=color, label=label)
    left += est[col].values
for i, t in enumerate(est["total_gb"]):
    ax.text(t + 0.15, i, f"{t:.1f} GB", va="center", fontsize=9)
for ram in (8, 16):
    ax.axvline(ram, color="k", ls=":", lw=1)
    ax.text(ram, -0.7, f"{ram} GB laptop", ha="center", fontsize=8)
ax.set_xlim(0, 17.5)
ax.set_xlabel("estimated RAM (GB)")
ax.set_title("Will it fit? Q4_K_M weights + KV cache + overhead")
ax.legend(loc="lower right", fontsize=8)
plt.show()

The KV cache is small at 4k. It is **linear in context length**, though, and long-context settings change the picture:

In [ ]:
contexts = np.array([2_048, 8_192, 32_768, 65_536, 131_072])
fig, ax = plt.subplots(figsize=(9, 3.6))
for name in ["3B", "7B", "14B"]:
    row = MODELS.loc[name]
    kv = [kv_cache_gb(row.layers, row.kv_heads, row.head_dim, c) for c in contexts]
    line, = ax.plot(contexts / 1024, kv, marker="o", label=f"{name} KV cache")
    ax.axhline(row.params_b * BYTES_PER_PARAM["Q4_K_M"], color=line.get_color(), ls="--", lw=1,
               label=f"{name} Q4 weights")
ax.set_xscale("log", base=2)
ax.set_xticks(contexts / 1024, [f"{c // 1024}k" for c in contexts])
ax.set_xlabel("context length (tokens)")
ax.set_ylabel("GB (fp16 KV cache)")
ax.set_title("KV cache grows linearly with context; at long contexts it can exceed the weights")
ax.legend(fontsize=7, ncol=2)
plt.show()
kv14 = kv_cache_gb(48, 8, 128, 131_072)
print(f"14B at 128k context: KV cache {kv14:.1f} GB vs Q4 weights {14.7 * 0.6:.1f} GB")
assert kv14 > 14.7 * 0.6

That is the whole "how much memory to serve" answer (`tr29`): weights are a fixed cost, and the KV cache scales with *context × concurrent requests*. So what you cut first is usually the KV cache: a shorter context, fewer parallel sequences, or KV-cache quantisation (in Ollama `OLLAMA_KV_CACHE_TYPE=q8_0`, which needs flash attention enabled).

## 6. Which model? Sizes at a glance

Approximate download sizes of the default (mostly Q4_K_M) tags, as listed on [ollama.com/library](https://ollama.com/library) at the time of writing. Check there before relying on them.

| Model (Ollama name) | Params | Download | Good for |
|---|---|---|---|
| `smollm2:135m` / `gemma3:270m` | 0.1-0.3 B | ~0.3 GB | plumbing tests only |
| `qwen2.5:0.5b` / `qwen3:0.6b` | 0.5-0.6 B | ~0.4-0.5 GB | trivial classification, very fast |
| `llama3.2:1b` / `gemma3:1b` | 1 B | ~0.8-1.3 GB | simple extraction and routing |
| `qwen2.5:1.5b` / `qwen3:1.7b` | 1.5-1.7 B | ~1.0-1.4 GB | light chat, summaries |
| **`qwen2.5:3b`** (these notebooks' default) / `llama3.2:3b` | 3 B | ~1.9-2.0 GB | decent chat, tool calling, JSON, on any laptop |
| `phi4-mini` / `qwen3:4b` / `gemma3:4b` | 3.8-4 B | ~2.5-3.3 GB | stronger reasoning for the size |
| `qwen2.5:7b` / `mistral` / `qwen3:8b` | 7-8 B | ~4.4-5.2 GB | good general quality, needs ~6+ GB RAM |
| `qwen2.5:14b` / `qwen3:14b` / `gemma3:12b` | 12-15 B | ~8-9 GB | near small-hosted-model quality, 16 GB+ RAM |
| `nomic-embed-text` | 0.14 B | ~0.3 GB | embeddings (768-dim) |

## 7. GGUF and quantisation levels

**GGUF** is llama.cpp's single-file model format: the weights (quantised in blocks, each with its own scale), the tokenizer, the chat template and the metadata, all in one file. "K-quants" (`_K_`) mix bit-widths per tensor, keeping the sensitive ones (e.g. attention output, embeddings) at higher precision.

| Level | ≈ bits/weight | ≈ bytes/param | Quality vs fp16 | When |
|---|---|---|---|---|
| `F16` / `BF16` | 16 | 2.0 | reference | GPUs with room to spare, or as the source for quantising |
| `Q8_0` | 8.5 | 1.06 | practically lossless | when memory allows |
| `Q6_K` | 6.6 | 0.82 | very close | a good "no-compromise" local choice |
| `Q5_K_M` | 5.7 | 0.71 | small loss | quality-sensitive work on modest hardware |
| **`Q4_K_M`** | 4.8 | 0.60 | small but measurable loss, reasoning degrades first | **the default**: the best size/quality balance |
| `Q3_K_M` / `Q2_K` | 3.9 / 3.4 | 0.49 / 0.42 | clearly worse | only to squeeze a bigger model in |

Rule of thumb: a bigger model at Q4 usually beats a smaller model at Q8 for the same memory. GPU-server formats differ (AWQ, GPTQ, FP8 for vLLM), but the trade-off is the same.

## 8. Other ways to run models locally

| Runtime | What it is | OpenAI-compatible server | Best for |
|---|---|---|---|
| **Ollama** | CLI + server wrapping llama.cpp, model registry | yes, `:11434/v1` | the easiest local dev loop |
| **LM Studio** | desktop GUI, model browser | yes, `:1234/v1` | trying models without a terminal |
| **llama.cpp** `llama-server` | the engine itself, any GGUF file | yes, `:8080/v1` | full control, embedded/edge, CPU |
| **vLLM** | GPU serving engine: PagedAttention, continuous batching | yes (`vllm serve <model>`) | **production** throughput on GPUs |
| **HF `transformers`** | Python library: load weights, `generate()` | no (it is a library) | research, fine-tuning, custom decoding |
| TGI / SGLang / TensorRT-LLM | other production GPU servers | yes | high-throughput serving |

The production point (`tl11`, `in13`): Ollama is a single-user developer tool, and concurrent requests largely queue. For many users on GPUs you want vLLM-class continuous batching, and your code doesn't change, because it's all the same OpenAI-compatible API.

## 9. Run a model yourself with Hugging Face `transformers`

No Ollama, no server: load `SmolLM2-135M-Instruct` (135 M parameters, ~270 MB, cached for these notebooks) and generate on the CPU. First look at what the model *actually* sees. The chat template turns role-tagged messages into one string with special tokens:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
torch.manual_seed(0)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
try:
    tok = AutoTokenizer.from_pretrained(NAME)
    lm = AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).eval()   # fp32: bf16 is slow on most CPUs
except Exception as e:
    raise RuntimeError(f"Could not load {NAME} (needs the Hugging Face cache or internet): {e}")

messages = [{"role": "system", "content": "You are a helpful assistant. Answer briefly."},
            {"role": "user", "content": "What is the capital of France?"}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
n_params = sum(p.numel() for p in lm.parameters())
print(f"parameters: {n_params / 1e6:.1f} M | fp32 weights: {lm.get_memory_footprint() / 1e6:.0f} MB")
assert prompt.rstrip().endswith("assistant")

`<|im_start|>role ... <|im_end|>` is this model's template (the "ChatML" style). Other families use different markers (Llama 3: `<|start_header_id|>`, Gemma: `<start_of_turn>`). Build the wrong string by hand and the model still answers, just worse, with no error (`tn24`). Always use `apply_chat_template`. Ollama and vLLM apply the model's own template for you, which is one reason to use them.

Now generate, timing every token with a stopping-criteria "clock":

In [ ]:
class TokenClock(StoppingCriteria):
    """Records a timestamp each time generate() produces a token; never stops generation."""
    def __init__(self):
        self.times = []
    def __call__(self, input_ids, scores, **kwargs):
        self.times.append(time.perf_counter())
        return False

def local_chat(user_text, max_new_tokens=24):
    ids = tok(tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=False,
                                      add_generation_prompt=True), return_tensors="pt")
    clock = TokenClock()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = lm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,          # greedy = repeatable
                          stopping_criteria=StoppingCriteriaList([clock]), pad_token_id=tok.eos_token_id)
    new = out[0, ids["input_ids"].shape[1]:]
    stamps = np.array(clock.times) - t0
    return tok.decode(new, skip_special_tokens=True).strip(), stamps, ids["input_ids"].shape[1]

local_chat("Hi", max_new_tokens=2)                                   # warm-up (first call pays one-off costs)
reply, stamps, n_prompt = local_chat("What is the capital of France?")
decode_rate = (len(stamps) - 1) / (stamps[-1] - stamps[0]) if len(stamps) > 1 else float("nan")
print("reply:", reply)
print(f"prompt {n_prompt} tokens | generated {len(stamps)} tokens | first token after {stamps[0] * 1000:.0f} ms | "
      f"decode {decode_rate:.1f} tokens/s")
assert "paris" in reply.lower()

In [ ]:
gaps = np.diff(np.concatenate([[0.0], stamps])) * 1000
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(1, len(gaps) + 1), gaps, color=["#e8590c"] + ["#1c7ed6"] * (len(gaps) - 1))
ax.set_xlabel("output token #")
ax.set_ylabel("ms for this token")
ax.set_title("Token 1 = prefill of the whole prompt + first step (orange); then one decode step per token")
plt.show()

Tokens per second (`in37`) means two different things. This is the **per-request decode rate**, which is what one reader experiences (humans read at roughly 5-10 tokens/s, so interactive use wants at least ~15). A server's **aggregate throughput** across many concurrent requests can be far higher, while each request gets slower. Always ask which one a number means. The 135 M model is tiny, and even so a laptop CPU gives it somewhere between a few and a few dozen tokens per second, depending on the CPU and whatever else is running. Every parameter is read from memory for every token, which is why decode is memory-bandwidth-bound.

## 10. Quality vs size

The same tiny model on two small tasks. (It already got the capital-of-France fact right above.)

In [ ]:
tasks = {"arithmetic": "What is 17 * 23? Give only the number.",
         "format": "Reply with only a JSON object with keys name and age for: Ana is 31 years old."}
answers = {}
for kind, q in tasks.items():
    answers[kind] = local_chat(q, max_new_tokens=20)[0]
    print(f"{kind:>10}: {answers[kind]!r}")
print("\narithmetic correct (17*23 = 391)?", "391" in answers["arithmetic"])
try:
    print("JSON parses?", json.loads(answers["format"][answers["format"].find("{"): answers["format"].rfind("}") + 1]))
except ValueError:
    print("JSON parses? no")

Run it and judge for yourself. Tiny models often get well-known facts and simple formats right but fail at arithmetic and multi-step reasoning, and they're easy to derail. Roughly:

| Size | Typical ability |
|---|---|
| < 1 B | pattern completion, simple classification, plumbing tests. Not trustworthy for facts |
| 1-4 B | usable chat, extraction, routing, basic tool calls, all better with few-shot examples and constrained output |
| 7-14 B | solid general assistant, decent code and reasoning, reliable JSON |
| 30-70 B+ | approaches hosted mid-tier models. Needs a serious GPU (or heavy quantisation) |

The practical pattern: prototype against a hosted model to learn what "good" looks like, then check whether a small local model (plus constrained decoding, few-shot, or a fine-tune for the narrow task) meets *your* eval set. Decide on numbers, not vibes ([02 · Prompt engineering](02_prompt_engineering.ipynb), section 7).

## Try it yourself

**1. Will it fit?** A team wants `qwen2.5:14b` at Q4_K_M with a 32k context on a 16 GB laptop, serving 2 conversations at once (each needs its own KV cache). Use `ram_gb`/`kv_cache_gb` to estimate total memory. Does it fit? What's the largest context that fits in 14 GB (leaving 2 GB for the OS)?

In [ ]:
# Solution — try it yourself first, then run this
row = MODELS.loc["14B"]
weights = row.params_b * BYTES_PER_PARAM["Q4_K_M"]
kv_32k = kv_cache_gb(row.layers, row.kv_heads, row.head_dim, 32_768)
total = weights + 2 * kv_32k + 0.5
print(f"weights {weights:.1f} GB + 2 x KV@32k {kv_32k:.1f} GB + overhead 0.5 = {total:.1f} GB -> fits in 16 GB? {total <= 16}")
budget = 14 - weights - 0.5
per_token_2seq = 2 * kv_cache_gb(row.layers, row.kv_heads, row.head_dim, 1)
print(f"largest context for 2 conversations in 14 GB: ~{int(budget / per_token_2seq):,} tokens each")
print(f"with a q8_0 KV cache (1 byte): ~{int(budget / (per_token_2seq / 2)):,} tokens each")

**2. What the model sees without a system prompt.** Render SmolLM2's chat template for a single user message with *no* system message. What system prompt does the template insert on its own? Why does that matter when you compare local and hosted models?

In [ ]:
# Solution — try it yourself first, then run this
bare = tok.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False, add_generation_prompt=True)
print(bare)
# Many templates inject a default system prompt (here: SmolLM's identity). Your own system prompt replaces it;
# forgetting that is one reason a local model "behaves differently" from the same prompt elsewhere.

**3. Greedy vs sampled.** Generate the answer to "Suggest a name for a coffee shop." twice with `do_sample=True, temperature=0.9, top_p=0.95` (different seeds) and twice greedy. Which settings are reproducible, and which would you use for a regression test?

In [ ]:
# Solution — try it yourself first, then run this
ids = tok(tok.apply_chat_template([{"role": "user", "content": "Suggest a name for a coffee shop."}],
                                  tokenize=False, add_generation_prompt=True), return_tensors="pt")
def gen(sample, seed):
    torch.manual_seed(seed)
    with torch.no_grad():
        out = lm.generate(**ids, max_new_tokens=6, do_sample=sample, pad_token_id=tok.eos_token_id,
                          **({"temperature": 0.9, "top_p": 0.95} if sample else {}))
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

greedy = [gen(False, s) for s in range(2)]
sampled = [gen(True, s) for s in range(2)]
print("greedy :", greedy)
print("sampled:", sampled)
assert len(set(greedy)) == 1          # greedy is deterministic: use it (or a fixed seed) in tests

## Say it in an interview

- **Self-host or API (30 s):** "Two independent reasons to self-host. Data that can't leave, which decides it outright. And steady high volume, where a busy GPU beats per-token pricing. Everything else favours an API: no idle cost, no on-call, better models on hard tasks. I'd do the crossover arithmetic: tokens per day × API price vs GPU-hours plus the engineers to run them."
- **Ollama:** a llama.cpp wrapper with a model registry and an OpenAI-compatible API. Great for development and private data on one machine. Not a production server for many users (no continuous batching), so vLLM fills that role, with the same client code.
- **Memory (the formula):** weights = params × bytes/param (fp16 2, Q8 ~1, Q4_K_M ~0.6), plus KV cache = 2 × layers × kv_heads × head_dim × context × 2 bytes × concurrent sequences, plus overhead. 7B at Q4 ≈ 4.5 GB of weights. The KV cache is what scales with traffic, so it's what you cut first.
- **Quantisation:** Q4_K_M is the default sweet spot. A bigger model at Q4 usually beats a smaller one at Q8 in the same memory. Reasoning and long-context tasks degrade first.
- **Gotchas they probe:** `max_completion_tokens` is ignored by Ollama (use `max_tokens`); the default context is 4k and longer prompts are silently truncated; `n`/`tool_choice` aren't supported; the first request pays the model-load cold start; chat templates matter, so use `apply_chat_template`.
- **Tokens/s:** ask "per request or aggregate, at what batch size?" Decode is memory-bandwidth-bound, while prefill is parallel and much faster per token.

## Next

- [Quantization int8 / int4 / NF4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb) · [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)
- [Tokenizers, models and generate()](../10_huggingface/02_tokenizers_models_and_generate.ipynb) · [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb)
- [LiteLLM gateway and model routing](../16_production/06_litellm_gateway_and_model_routing.ipynb): one API across OpenAI, Ollama and more
- [06 · Conversation memory and context windows](06_conversation_memory_and_context_windows.ipynb)
- Theory: [ai_tooling course](../../ai_tooling/index.html) · [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)